# RecSys Beauty — end-to-end demo

Amazon Reviews 2023 / All_Beauty: support-фильтр `rating>=3`, пользователь >=3 взаимодействий, товар >=5. Temporal split: train / valid / test.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

from src.data_utils import load_processed
from src.predict import load_baseline_bundle, load_two_tower_recommender, recommend_for_user, recommendations_frame

interactions, items, users = load_processed()
print(interactions['split'].value_counts())
print('users', users.shape[0], 'items', items.shape[0])

In [ ]:
prep = json.loads(Path('reports/data_prep_report.json').read_text(encoding='utf-8'))
prep

In [ ]:
baselines = load_baseline_bundle()
two_tower = load_two_tower_recommender()
print('methods', list(baselines['recommenders'].keys()))
print('two_tower', two_tower is not None)

In [ ]:
# Pick one of the most active users and compare several methods.
top_user = interactions.groupby('user_idx').size().idxmax()
user_id = users.loc[users['user_idx'] == top_user, 'user_id'].iloc[0]
print('user', user_id, 'user_idx', top_user)

for method in ['popularity', 'content_tfidf', 'item_knn', 'hybrid', 'two_tower']:
    try:
        rec_items, scores = recommend_for_user(method, int(top_user), interactions, items, users, k=5, baseline_bundle=baselines, two_tower=two_tower)
        df = recommendations_frame(rec_items, scores, items)
        print('\n', method)
        print(df[['rank', 'title', 'brand', 'score']].head().to_string(index=False))
    except Exception as exc:
        print(method, 'failed:', exc)

In [ ]:
# Offline metrics from reports/*.csv
frames = []
for path in [Path('reports/baseline_metrics.csv'), Path('reports/two_tower_metrics.csv')]:
    if path.exists():
        frames.append(pd.read_csv(path))
metrics = pd.concat(frames, ignore_index=True)
test_metrics = metrics[metrics['eval_split'] == 'test'].sort_values('recall@10', ascending=False)
test_metrics[['method', 'recall@10', 'ndcg@10', 'hit@10', 'coverage', 'novelty']]

In [ ]:
cols = [c for c in ['method', 'recall@10', 'ndcg@10', 'hit@10', 'coverage', 'novelty'] if c in test_metrics.columns]
ax = test_metrics[cols].set_index('method').plot(kind='bar', subplots=True, legend=False, figsize=(10, 8))
plt.tight_layout()
plt.show()